# Construeix el dataset analític aplanat

## 0. Imports necessaris

In [59]:
import os
import pandas as pd

## 1. Carregar les taules

In [60]:
print("Cargando cohort.xlsx ...")
cohort = pd.read_excel("raw/cohort.xlsx")
print(f"  cohort: {cohort.shape[0]} filas, {cohort.shape[1]} columnas")

print("Cargando diagnostics.xlsx ...")
diagnostics = pd.read_excel("raw/diagnostics.xlsx")
print(f"  diagnostics: {diagnostics.shape[0]} filas, {diagnostics.shape[1]} columnas")

print("Cargando farmacs.xlsx ...")
farmacs = pd.read_excel("raw/farmacs.xlsx")
print(f"  farmacs: {farmacs.shape[0]} filas, {farmacs.shape[1]} columnas")

Cargando cohort.xlsx ...
  cohort: 37902 filas, 5 columnas
Cargando diagnostics.xlsx ...
  diagnostics: 37902 filas, 10 columnas
Cargando farmacs.xlsx ...
  farmacs: 23580 filas, 16 columnas


### 1.1 Eliminem informació irrellevant per l'estudi

In [61]:
# Eliminem la situació del pacient (mort o viu)
cohort = cohort.drop(columns="situacio")

## 2. Ajuntem les taules
### 2.2 Left Join 1:1

In [62]:
print("\nRealizando LEFT JOIN cohort ← diagnostics (on id_pacient) ...")
df = cohort.merge(diagnostics, on="id_pacient", how="left")
print(f"  Resultado parcial: {df.shape[0]} filas, {df.shape[1]} columnas")

print("Realizando LEFT JOIN resultado ← farmacs (on id_pacient) ...")
df = df.merge(farmacs, on="id_pacient", how="left")
print(f"  Resultado parcial: {df.shape[0]} filas, {df.shape[1]} columnas")


Realizando LEFT JOIN cohort ← diagnostics (on id_pacient) ...
  Resultado parcial: 37902 filas, 13 columnas
Realizando LEFT JOIN resultado ← farmacs (on id_pacient) ...
  Resultado parcial: 37902 filas, 28 columnas


In [63]:
# Emplenem els valors NaN amb zeros, que en aquest moment solament son les columnes de farmacs.
# Assumim que si és NaN, no tenen farmacs
df = df.fillna(0)

## 3. Agregacions 1:N — recompte de visites per pacient
### 3.1 Visites Hospital (Intervals de dies)

In [64]:
print("\nCargant raw/visites_hospital.xlsx ...")
hosp = pd.read_excel("raw/visites_hospital.xlsx")
print(f"  visites_hospital: {hosp.shape[0]} files")

# Comptem les visites per a cada interval de dies 1-121, 122-242, 243-365
hosp_1_121 = hosp[hosp["data"].between(1, 121)].groupby("id_pacient").size().rename("visites_hosp_1_121")
hosp_122_242 = hosp[hosp["data"].between(122, 242)].groupby("id_pacient").size().rename("visites_hosp_122_242")
hosp_243_365 = hosp[hosp["data"].between(243, 365)].groupby("id_pacient").size().rename("visites_hosp_243_365")

# Ajuntem les 3 columnes noves al dataset principal
for col, series in [("visites_hosp_1_121", hosp_1_121), 
                    ("visites_hosp_122_242", hosp_122_242), 
                    ("visites_hosp_243_365", hosp_243_365)]:
    df = df.merge(series, on="id_pacient", how="left")
    df[col] = df[col].fillna(0).astype(int)
    print(f"  Columna '{col}' afegida.")



Cargant raw/visites_hospital.xlsx ...
  visites_hospital: 10163 files
  Columna 'visites_hosp_1_121' afegida.
  Columna 'visites_hosp_122_242' afegida.
  Columna 'visites_hosp_243_365' afegida.


### 3.2 Visites Atenció Intermèdia (Intervals de dies)

In [65]:
print("\nCargando raw/visites_intermedia.xlsx ...")
inter = pd.read_excel("raw/visites_intermedia.xlsx")
print(f"  visites_intermedia: {inter.shape[0]} files")

# Comptem les visites per a cada interval de dies
inter_1_121 = inter[inter["data"].between(1, 121)].groupby("id_pacient").size().rename("visites_inter_1_121")
inter_122_242 = inter[inter["data"].between(122, 242)].groupby("id_pacient").size().rename("visites_inter_122_242")
inter_243_365 = inter[inter["data"].between(243, 365)].groupby("id_pacient").size().rename("visites_inter_243_365")

# Ajuntem les 3 columnes noves al dataset principal
for col, series in [("visites_inter_1_121", inter_1_121), 
                    ("visites_inter_122_242", inter_122_242), 
                    ("visites_inter_243_365", inter_243_365)]:
    df = df.merge(series, on="id_pacient", how="left")
    df[col] = df[col].fillna(0).astype(int)
    print(f"  Columna '{col}' afegida.")


Cargando raw/visites_intermedia.xlsx ...
  visites_intermedia: 2465 files
  Columna 'visites_inter_1_121' afegida.
  Columna 'visites_inter_122_242' afegida.
  Columna 'visites_inter_243_365' afegida.


### 3.3 Visites Urgències (Risc Vital Potencial)

In [66]:
print("\nCargando raw/visites_urgencies.xlsx ...")
urg = pd.read_excel("raw/visites_urgencies.xlsx")
print(f"  visites_urgencies: {urg.shape[0]} files")

# Filtrem només les files amb el nivell de triatge requerit i comptem per pacient
urg_risc = urg[urg["nivell_triatge"] == "Risc vital potencial"].groupby("id_pacient").size().rename("visites_urgencies_risc_vital")

df = df.merge(urg_risc, on="id_pacient", how="left")
df["visites_urgencies_risc_vital"] = df["visites_urgencies_risc_vital"].fillna(0).astype(int)
print("  Columna 'visites_urgencies_risc_vital' afegida.")


Cargando raw/visites_urgencies.xlsx ...
  visites_urgencies: 19529 files
  Columna 'visites_urgencies_risc_vital' afegida.


### 3.4 Visites Atenció Primària (Total de visites)

In [67]:
print("\nCargando raw/visites_primaria.xlsx ...")
primaria = pd.read_excel("raw/visites_primaria.xlsx")
print(f"  visites_primaria: {primaria.shape[0]} filas")

# Sumem la columna "visites" per pacient
prim_visites = primaria.groupby("id_pacient")["visites"].sum().rename("num_visitas_primaria")

df = df.merge(prim_visites, on="id_pacient", how="left")
df["num_visitas_primaria"] = df["num_visitas_primaria"].fillna(0).astype(int)
print("  Columna 'num_visitas_primaria' añadida.")


Cargando raw/visites_primaria.xlsx ...
  visites_primaria: 98633 filas
  Columna 'num_visitas_primaria' añadida.


## 4. Variables de laboratori — mean i slope per prova

In [68]:
print("\nCargando laboratori.xlsx ...")
lab = pd.read_excel("raw/laboratori.xlsx")
print(f"  laboratori: {lab.shape[0]} files, {lab['id_pacient'].nunique()} pacients, {lab['desc_prova_ics'].nunique()} proves")


# Nom curt per a cada prova (per utilitzar com a sufix de columna)
SHORT_NAMES = {
    "GLUCOSA-SÈRUM":                                  "glucosa",
    "UREA-SÈRUM":                                     "urea",
    "PROTEÏNA C REACTIVA (PCR)-SÈRUM":                "pcr",
    "BILIRUBINA-SÈRUM":                               "bilirubina",
    "ASPARTAT AMINOTRANSFERASA-SÈRUM":                "ast",
    "ALANINA AMINOTRANSFERASA-SÈRUM":                 "alt",
    "ALBÚMINA-SÈRUM":                                 "albumina",
    "COLESTEROL-SÈRUM":                               "colesterol",
    "FOSFATASA ALCALINA-SÈRUM":                       "fosfatasa",
    "PROTEÏNA-SÈRUM":                                 "proteina",
    "TIROTROPINA-SÈRUM":                              "tsh",
    "ERITROSEDIMENTACIÓ (VSG)-SANG":                  "vsg",
    "ÀCID FÒLIC-SÈRUM":                               "ac_folico",
    "COBALAMINES (VITAMINA B12)-SÈRUM":               "vit_b12",
    "FERRITINA-SÈRUM":                                "ferritina",
    "FERRO-SÈRUM":                                    "ferro",
    "PRO-BNP-SÈRUM":                                  "pro_bnp",
    "DÍMER D DE LA FIBRINA (IMMUNOTURBIDIMETRIA)-PLASMA": "dimero_d",
}

lab["prova_short"] = lab["desc_prova_ics"].map(SHORT_NAMES)


Cargando laboratori.xlsx ...
  laboratori: 15282 files, 3312 pacients, 18 proves


In [69]:
lab

,id_pacient,desc_prova_ics,mean,median,std,slope,id_test_lab,prova_short
0,5,GLUCOSA-SÈRUM,95.00,95.00,NaN,NaN,5GLUCOSA-SÈ,glucosa
1,5,UREA-SÈRUM,30.20,30.20,NaN,NaN,5UREA-SÈRUM,urea
2,15,UREA-SÈRUM,47.50,47.50,NaN,NaN,15UREA-SÈRUM,urea
3,15,PROTEÏNA C REACTIVA (PCR)-SÈRUM,2.13,2.13,NaN,NaN,15PROTEÏNA C,pcr
4,15,BILIRUBINA-SÈRUM,0.50,0.50,NaN,NaN,15BILIRUBINA,bilirubina
...,...,...,...,...,...,...,...,...
15277,37893,PROTEÏNA C REACTIVA (PCR)-SÈRUM,0.64,0.64,NaN,NaN,37893PROTEÏNA C,pcr
15278,37893,GLUCOSA-SÈRUM,260.00,260.00,NaN,NaN,37893GLUCOSA-SÈ,glucosa
15279,37901,PROTEÏNA C REACTIVA (PCR)-SÈRUM,0.40,0.40,NaN,NaN,37901PROTEÏNA C,pcr
15280,37901,UREA-SÈRUM,44.80,44.80,NaN,NaN,37901UREA-SÈRUM,urea


In [70]:
lab[lab["slope"].notna()]

,id_pacient,desc_prova_ics,mean,median,std,slope,id_test_lab,prova_short
6,32,UREA-SÈRUM,33.600000,33.600,3.818377,0.040602,32UREA-SÈRUM,urea
9,49,BILIRUBINA-SÈRUM,0.633333,0.800,0.378594,-0.180769,49BILIRUBINA,bilirubina
10,49,ASPARTAT AMINOTRANSFERASA-SÈRUM,28.000000,24.000,8.717798,3.923077,49ASPARTAT A,ast
11,49,PROTEÏNA C REACTIVA (PCR)-SÈRUM,22.810000,21.335,6.619743,3.765556,49PROTEÏNA C,pcr
12,49,UREA-SÈRUM,58.100000,59.550,13.741179,-5.088889,49UREA-SÈRUM,urea
...,...,...,...,...,...,...,...,...
15271,37860,UREA-SÈRUM,175.550000,175.550,36.557421,-1.723333,37860UREA-SÈRUM,urea
15272,37870,ASPARTAT AMINOTRANSFERASA-SÈRUM,17.500000,17.500,2.121320,-0.088235,37870ASPARTAT A,ast
15273,37870,UREA-SÈRUM,22.650000,22.650,2.050610,-0.085294,37870UREA-SÈRUM,urea
15274,37870,PROTEÏNA C REACTIVA (PCR)-SÈRUM,0.120000,0.120,0.042426,0.001765,37870PROTEÏNA C,pcr


In [71]:
# Pivotar: una columna per prova × mean
lab_mean = lab.pivot_table(index="id_pacient", columns="prova_short", values="mean").add_suffix("_mean")

# CREEM ELS INDICADORS: Indiquem si el valor de la mitjana és NaN o no, és a dir si s'ha fet la prova o no, convertim True/False a 1/0
lab_done = lab_mean.notna().astype(int)
lab_done.columns = [f"{c}_realitzada" for c in lab_done.columns]

# Emplenem els valors nulls amb la moda de cada columna i afegim l'indicador
lab_mean = lab_mean.fillna(lab_mean.mode().iloc[0])
lab_mean = lab_mean.join(lab_done)
print (lab_mean.columns)

Index(['ac_folico_mean', 'albumina_mean', 'alt_mean', 'ast_mean',
       'bilirubina_mean', 'colesterol_mean', 'dimero_d_mean', 'ferritina_mean',
       'ferro_mean', 'fosfatasa_mean', 'glucosa_mean', 'pcr_mean',
       'pro_bnp_mean', 'proteina_mean', 'tsh_mean', 'urea_mean',
       'vit_b12_mean', 'vsg_mean', 'ac_folico_mean_realitzada',
       'albumina_mean_realitzada', 'alt_mean_realitzada',
       'ast_mean_realitzada', 'bilirubina_mean_realitzada',
       'colesterol_mean_realitzada', 'dimero_d_mean_realitzada',
       'ferritina_mean_realitzada', 'ferro_mean_realitzada',
       'fosfatasa_mean_realitzada', 'glucosa_mean_realitzada',
       'pcr_mean_realitzada', 'pro_bnp_mean_realitzada',
       'proteina_mean_realitzada', 'tsh_mean_realitzada',
       'urea_mean_realitzada', 'vit_b12_mean_realitzada',
       'vsg_mean_realitzada'],
      dtype='str')


In [72]:
# Pivotar: una columna per prova × slope
lab_slope = lab.pivot_table(index="id_pacient", columns="prova_short", values="slope").add_suffix("_slope")

# CREEM ELS INDICADORS: Indiquem si el valor de la mitjana és NaN o no, és a dir si s'ha fet la prova o no, convertim True/False a 1/0
slope_done = lab_slope.notna().astype(int)
slope_done.columns = [f"{c}_registrat" for c in slope_done.columns]

# Emplenem els valors nulls amb la moda de cada columna i afegim l'indicador
lab_slope = lab_slope.fillna(lab_slope.mode().iloc[0])
lab_slope = lab_slope.join(slope_done)
print (lab_slope.columns)

Index(['ac_folico_slope', 'albumina_slope', 'alt_slope', 'ast_slope',
       'bilirubina_slope', 'colesterol_slope', 'ferritina_slope',
       'ferro_slope', 'fosfatasa_slope', 'glucosa_slope', 'pcr_slope',
       'pro_bnp_slope', 'proteina_slope', 'tsh_slope', 'urea_slope',
       'vit_b12_slope', 'vsg_slope', 'ac_folico_slope_registrat',
       'albumina_slope_registrat', 'alt_slope_registrat',
       'ast_slope_registrat', 'bilirubina_slope_registrat',
       'colesterol_slope_registrat', 'ferritina_slope_registrat',
       'ferro_slope_registrat', 'fosfatasa_slope_registrat',
       'glucosa_slope_registrat', 'pcr_slope_registrat',
       'pro_bnp_slope_registrat', 'proteina_slope_registrat',
       'tsh_slope_registrat', 'urea_slope_registrat',
       'vit_b12_slope_registrat', 'vsg_slope_registrat'],
      dtype='str')


In [73]:
# Ajuntem tot en una sola taula
lab_pivot = lab_mean.join(lab_slope).reset_index()
print(f"  Columnas de laboratorio generadas: {lab_pivot.shape[1] - 1}")

# Left join
df = df.merge(lab_pivot, on="id_pacient", how="left")
print(f"  Dataset tras añadir laboratorio: {df.shape[0]} filas, {df.shape[1]} columnas")

  Columnas de laboratorio generadas: 70
  Dataset tras añadir laboratorio: 37902 filas, 106 columnas


## 5. Resum

In [75]:
print(f"\n{'='*50}")
print(f"=== Dataset resultant ===")
print(f"  Files:    {df.shape[0]}")
print(f"  Columnes: {df.shape[1]}")
print(f"  Noms de les columnes: {list(df.columns)}")

print(f"\n  Estadístiques de visites:")
visit_cols = [
    "visites_hosp_1_121", "visites_hosp_122_242", "visites_hosp_243_365",
    "visites_inter_1_121", "visites_inter_122_242", "visites_inter_243_365",
    "visites_urgencies_risc_vital", "num_visitas_primaria"
]
for col in visit_cols:
    if col in df.columns:
        print(f"    {col}: mitjana={df[col].mean():.2f}, màx={df[col].max()}")

# Identifiquem les columnes de laboratori (incloses les de control/realització si cal)
lab_cols = [c for c in df.columns if any(c.endswith(s) for s in ["_mean", "_slope", "_realitzada", "_registrat"])]

print(f"\n  Cobertura de laboratori (pacients amb dades):")
for col in sorted(lab_cols):
    non_null = df[col].notna().sum()
    print(f"    {col}: {non_null} ({100*non_null/len(df):.1f}%)")

print(f"\n  Valors nuls per columna (no-lab):")
nulls = df.isnull().sum()
for col in df.columns:
    if nulls[col] > 0 and col not in lab_cols:
        print(f"    {col}: {nulls[col]} nuls ({100*nulls[col]/len(df):.1f}%)")



=== Dataset resultant ===
  Files:    37902
  Columnes: 106
  Noms de les columnes: ['id_pacient', 'sexe', 'cronic', 'grup_edat', 'altres', 'problemes_salut_aguts', 'problemes_salut_anomalia_congenita', 'problemes_salut_cronics', 'problemes_salut_indefinits', 'problemes_salut_neoplasia_benigna', 'problemes_salut_neoplasia_maligna', 'signes_i_sintomes', 'diags_totals', 'antiinfecciosos_per_a_us_sistemic', 'antineoplasics_i_immunomoduladors', 'dermatologics', 'diversos', 'preparats_hormonals_sistemics', 'productes_antiparasitaris,_insecticides_i_repellents', 'sang_i_organs_hematopoetics', 'sistema_cardiovascular', 'sistema_digestiu_i_metabolisme', 'sistema_genitourinari_i_hormones_sexuals', 'sistema_musculoesqueletic', 'sistema_nervios', 'sistema_respiratori', 'organs_dels_sentits', 'farmacs_totals', 'visites_hosp_1_121', 'visites_hosp_122_242', 'visites_hosp_243_365', 'visites_inter_1_121', 'visites_inter_122_242', 'visites_inter_243_365', 'visites_urgencies_risc_vital', 'num_visitas_p

## 6. Desar

In [76]:
output_path = "processed/dataset_analitico.xlsx"
print(f"\nGuardando en {output_path} ...")
df.to_excel(output_path, index=False)
print(f"¡Guardado correctamente! ({output_path})")


Guardando en processed/dataset_analitico.xlsx ...
¡Guardado correctamente! (processed/dataset_analitico.xlsx)
